## Cargar el CSV limpio

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("secop_ii_agencia_logistica_limpio.csv")

cols_fecha = [c for c in df.columns if c.startswith('fecha') or c == 'ultima_actualizacion']
for c in cols_fecha:
    df[c] = pd.to_datetime(df[c], errors='coerce')

print(df.shape)

(8051, 88)


Variables de tiempo

In [2]:
df['anio_firma'] = df['fecha_de_firma'].dt.year
df['mes_firma'] = df['fecha_de_firma'].dt.month
df['trimestre_firma'] = df['fecha_de_firma'].dt.quarter
df['nombre_mes_firma'] = df['fecha_de_firma'].dt.month_name()

Tratamiento del valor del contrato

In [3]:
df['valor_log'] = np.log1p(df['valor_del_contrato'].clip(lower=0))

q1, q3 = df['valor_log'].quantile([0.25, 0.75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr
df['es_outlier_valor'] = df['valor_log'] > limite_superior

p995 = df['valor_del_contrato'].quantile(0.995)
df['valor_contrato_capped'] = df['valor_del_contrato'].clip(upper=p995)

Ejecución financiera

In [5]:
df['pct_pagado'] = np.where(
    df['valor_del_contrato'] > 0,
    (df['valor_pagado'] / df['valor_del_contrato']).round(4),
    np.nan
)

Banderas de estado

In [ ]:
df['esta_liquidado'] = df['fecha_fin_liquidacion'].notna()
df['fue_prorrogado'] = df['fecha_de_notificacion_de_prorrogacion'].notna()